# Testing the Fine-Tuned XLSR on the Test Sets


**Author:**

Salvador Wahnon Palma s2665070@u.tsukuba.ac.jp

University of Tsukuba / Interaction Lab


<br>

**Objective:**

Evaluate the phoneme recognizer fine-tuned in `FineTuning.ipynb` on the held-out US + JP
test sets, and expose a single `Transcribe(audio)` function that maps a waveform to its
predicted IPA phoneme sequence.

---

# 1. Setup Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [4]:
!pip uninstall -y -q datasets 2>/dev/null
!pip install -q -U "transformers>=4.44" jiwer soundfile "pyarrow>=17,<19"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 103.3 MB/s eta 0:00:0000:010:01


# 2. Load model + processor and test data

## 2.0 Imports and paths

In [5]:
import importlib.util
if importlib.util.find_spec("datasets") is not None:
    raise RuntimeError("datasets installed. Uninstall first")


import io
import numpy as np
import pandas as pd
import soundfile as sf
import torch

MODEL_DIR = "/content/drive/MyDrive/Tsukuba/Datasets and Models/xlsr-jp-us-ipa/checkpoint-1200"
DATA_DIR = "/content/drive/MyDrive/Tsukuba/Datasets and Models"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_DIR).to(device)
model.eval()

tokenizer = processor.tokenizer
DELIM = tokenizer.word_delimiter_token

print("Loaded model + processor from:", MODEL_DIR)
print("Vocab size:", len(tokenizer), " delimiter:", repr(DELIM))

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loaded model + processor from: /content/drive/MyDrive/Tsukuba/Datasets and Models/xlsr-jp-us-ipa/checkpoint-1200
Vocab size: 75  delimiter: '|'


## 2.1 IPA decoding + the `Transcribe` function

Same id → phoneme mapping as training (`IdsToIpa`): the tokenizer's `decode()` strips the
delimiter and space-joins, which would glue multi-char phonemes, so we map ids to tokens
directly. `Transcribe(audio, sr)` takes a waveform (np.array) and returns the predicted
space-separated IPA string.

In [7]:
SPECIAL_TOKENS = {tokenizer.pad_token, tokenizer.unk_token, getattr(tokenizer, "bos_token", None), getattr(tokenizer, "eos_token", None)}

def IdsToIpa(ids, group_tokens: bool = True) -> str:
    ids = [int(i) for i in ids]
    if group_tokens:
        ids = [i for j, i in enumerate(ids) if j == 0 or i != ids[j - 1]]
        
    tokens = tokenizer.convert_ids_to_tokens(ids)
    phonemes = [t for t in tokens if t not in SPECIAL_TOKENS and t != DELIM]
    return " ".join(phonemes)


@torch.no_grad()
def Transcribe(audio, sr=16000) -> str:
    
    if isinstance(audio, dict):
        audio, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
    
    audio = np.asarray(audio, dtype=np.float32)
    if audio.ndim > 1: #If stero put to mono
        audio = audio.mean(axis=1)
        
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_values.to(device)
    logits = model(inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)[0]
    
    return IdsToIpa(pred_ids, group_tokens=True)

## 2.2 Load the test sets

Same schema as train/validation: `audio` (dict with WAV `bytes`) + space-separated `IPA`.
We tag each with `lang` and concatenate US + JP.

In [9]:
from random import random


us_test = pd.read_parquet(f"{DATA_DIR}/US_test.parquet")
jp_test = pd.read_parquet(f"{DATA_DIR}/JP_test.parquet")

us_test = us_test[["audio", "IPA"]].copy()
us_test["lang"] = "us"
jp_test = jp_test[["audio", "IPA"]].copy()
jp_test["lang"] = "jp"

test_df = pd.concat([us_test, jp_test], ignore_index=True)

print("US test:", len(us_test), "| JP test:", len(jp_test), "| total:", len(test_df))



US test: 670 | JP test: 882 | total: 1552


# 3. Evaluate PER on the test sets

PER (Phoneme Error Rate) = word-error-rate over space-separated phonemes. We report overall
and per-language, since US (TIMIT) and JP (JVS) are not directly comparable.

In [10]:
import jiwer
from tqdm.auto import tqdm

transcriptions = {"fine-tuned" : []}

def TranscribeAll(df, desc="test"):
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        ref = row["IPA"]
        transcriptions["fine-tuned"].append(Transcribe(row["audio"]))


TranscribeAll(test_df)
print(len(transcriptions["fine-tuned"]))

  0%|          | 0/1552 [00:00<?, ?it/s]

1552


In [11]:
import json
with open(f"{DATA_DIR}/transcriptions_finetuned.json", "w", encoding="utf-8") as f:
    json.dump(transcriptions, f, ensure_ascii=False, indent=4)

In [10]:
# Per-language PER.
for lang in ["us", "jp"]:
    subset = test_df[test_df["lang"] == lang].reset_index(drop=True)
    per, _, _ = EvaluatePER(subset, desc=lang)
    print(f"{lang.upper()} test PER: {per:.4f}  (n={len(subset)})")

us:   0%|          | 0/670 [00:00<?, ?it/s]

US test PER: 0.1261  (n=670)


jp:   0%|          | 0/882 [00:00<?, ?it/s]

JP test PER: 0.0161  (n=882)
